# Task 3 · Training on Colab (T3-08 and T3-10)

Based on `common/colab_remote_gpu_setup.ipynb` (SETUP-05). Sections 1-5 set up
the session; section 6 holds one cell per T3-08 run and section 8 the T3-10
fine-tuning runs (section 6 is not needed for T3-10).

**Before running** (once):
1. **Runtime > Change runtime type > T4 GPU**.
2. The team folder `ml_tek5_checkpoints` is in your **My Drive** (or a shortcut
   to it, see the SETUP-05 notebook, section 2).
3. `ml_tek5_task3_data.zip`, made locally by
   `uv run python -m task3_uvr.pack_data`, is uploaded to
   `ml_tek5_checkpoints/task3/`.
4. A Colab secret `GITHUB_TOKEN` (fine-grained token, read-only `Contents` on
   the team repo) with notebook access on.
5. `BRANCH` below is pushed to GitHub.
6. For T3-10 only: `UVR-MDX-NET-Inst_HQ_3.onnx` is uploaded to
   `ml_tek5_checkpoints/task3/models/`.

Everything that must survive a disconnect goes to Drive, under
`ml_tek5_checkpoints/task3/`: checkpoints, loss logs, exported ONNX models and
the results CSVs and figures. After a disconnect, rerun sections 1-5, then the
run cell that was interrupted: it resumes from its last checkpoint (every 500
steps for T3-08, 250 for T3-10). A finished run cell can be rerun safely; it only exports and measures
the SDR again.

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Drive

Same folder as SETUP-05. Task 3 files go to its `task3/` subfolder.

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")

CHECKPOINT_DIR = "/content/drive/MyDrive/ml_tek5_checkpoints"
if not os.path.isdir(CHECKPOINT_DIR):
    raise FileNotFoundError(
        f"{CHECKPOINT_DIR} not found: add the team folder (or a shortcut named "
        "ml_tek5_checkpoints) to My Drive first"
    )
TASK3_DIR = f"{CHECKPOINT_DIR}/task3"
SMALL_NN_DIR = f"{TASK3_DIR}/small_nn"  # T3-08 checkpoints, logs and ONNX models
DRIVE_RESULTS = f"{TASK3_DIR}/results"  # copies of results/task3_* files
for path in (TASK3_DIR, SMALL_NN_DIR, DRIVE_RESULTS):
    os.makedirs(path, exist_ok=True)
print("Task 3 folder:", TASK3_DIR)

## 3. Clone the repository

As in SETUP-05: the token is sent as a one-off header, never written to
`.git/config`. Set `BRANCH` to the branch that holds the code to run (the
T3-08 branch until it is merged, then `master`).

In [ ]:
import base64
import getpass

# T3-10's branch, which also holds T3-08's code.
BRANCH = "28-t3-10-run-a-first-fine-tuning-on-degraded-audio"

try:
    from google.colab import userdata

    token = userdata.get("GITHUB_TOKEN")
except Exception:  # noqa: BLE001 - no secret, no access or not Colab: ask below
    token = None
if not token:
    token = getpass.getpass("GitHub token (read access to the team repo): ")

REPO_URL = "https://github.com/NattanCochet/G-AIA-900-PAR-9-1-machinelearning-11.git"
REPO_DIR = "/content/ml-tek5-project"
basic_auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
AUTH_HEADER = f"AUTHORIZATION: basic {basic_auth}"

# A second Authorization header (Colab's or an earlier run's) makes GitHub
# answer 400 "Duplicate header", so clear them first.
!git config --global --unset-all http.https://github.com/.extraheader 2>/dev/null
!git config --system --unset-all http.https://github.com/.extraheader 2>/dev/null

if not os.path.isdir(REPO_DIR):
    !git -c credential.helper= -c http.extraheader="{AUTH_HEADER}" clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} -c credential.helper= -c http.extraheader="{AUTH_HEADER}" fetch origin {BRANCH}
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} -c credential.helper= -c http.extraheader="{AUTH_HEADER}" pull --ff-only origin {BRANCH}
!git -C {REPO_DIR} log --oneline -1

## 4. Dependencies

As in SETUP-05: Colab's CUDA PyTorch is kept (`uv sync` would install the CPU
wheel pinned for the laptops); only the task 3 packages Colab lacks are
installed. Keep this list in sync with `pyproject.toml`.

In [ ]:
!pip install -q soundfile onnx onnxruntime onnx2pytorch mir_eval

In [ ]:
import importlib
import sys

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()

import torch

assert torch.cuda.is_available(), "GPU not enabled: Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

## 5. Data

The zip is copied from Drive to the local disk and unzipped to
`/content/data` (reading WAV files from Drive is slow). This takes a few
minutes and is skipped when the data is already there.

In [ ]:
import shutil
import subprocess

DATA_ZIP = f"{TASK3_DIR}/ml_tek5_task3_data.zip"
DATA_DIR = "/content/data"
MUSDB7_DIR = f"{DATA_DIR}/musdb7"

if not os.path.isdir(f"{DATA_DIR}/musdb7_narrowband/test"):
    if not os.path.isfile(DATA_ZIP):
        raise FileNotFoundError(f"Upload ml_tek5_task3_data.zip to {TASK3_DIR} first")
    os.makedirs(DATA_DIR, exist_ok=True)
    shutil.copy(DATA_ZIP, "/content/data.zip")
    subprocess.run(["unzip", "-q", "-o", "/content/data.zip", "-d", DATA_DIR], check=True)
    os.remove("/content/data.zip")

os.environ["ML_TEK5_MUSDB7_DIR"] = MUSDB7_DIR
for dataset in ("musdb7", "musdb7_phone", "musdb7_narrowband"):
    counts = {s: len(os.listdir(f"{DATA_DIR}/{dataset}/{s}")) for s in ("train", "test")}
    print(dataset, counts)  # expected: 94 train, 50 test

### Results on Drive

The training scripts add rows to `results/task3_sdr_clips.csv`,
`results/task3_uvr.csv`, `results/task3_finetune.json` and
`results/task3_finetune_before_after.csv` in the clone, which a disconnect
wipes. Before each
run, the Drive copies (if any) replace the clone's; after it, every
`results/task3_*` file is copied back to Drive. Drive therefore holds the full
history across sessions.

In [ ]:
import glob

# Files the scripts add to rather than rewrite: pulled before a run so that
# push_results() never replaces Drive's copy with a shorter one.
RESULT_CSVS = (
    "task3_sdr_clips.csv",
    "task3_uvr.csv",
    "task3_finetune.json",
    "task3_finetune_before_after.csv",
)


def pull_results():
    for name in RESULT_CSVS:
        if os.path.exists(f"{DRIVE_RESULTS}/{name}"):
            shutil.copy(f"{DRIVE_RESULTS}/{name}", f"{REPO_DIR}/results/{name}")


def push_results():
    for path in glob.glob(f"{REPO_DIR}/results/task3_*"):
        shutil.copy(path, DRIVE_RESULTS)
    print("results copied to", DRIVE_RESULTS)


def python_module(*args):
    """Run `python -m ...` from the repository root, print its output as it
    comes, raise if it fails."""
    cmd = [sys.executable, "-u", "-m", *map(str, args)]
    with subprocess.Popen(
        cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    ) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
    if proc.returncode:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

## 6. T3-08: small network from scratch

Four runs: {musdb7, musdb7_phone} x {remix, no remix}. Each one trains
`STEPS` steps of batch 8, saves its checkpoint and loss log to
`task3/small_nn/` on Drive every 500 steps, exports the ONNX model and its UVR
settings there, and measures the validation and test SDR (test for reporting
only). `--resume` is always passed: a run with no checkpoint starts from
step 0.

To train longer after a run has finished, raise `STEPS` and rerun its cell:
it resumes from the final checkpoint.

In [ ]:
STEPS = 4000
BATCH_SIZE = 8


def small_nn(dataset, remix):
    name = f"SmallMaskNet_{dataset}_{'remix' if remix else 'noremix'}"
    pull_results()
    try:
        python_module(
            "task3_uvr.train_small_nn",
            "--dataset", dataset,
            "--remix" if remix else "--no-remix",
            "--steps", STEPS,
            "--batch-size", BATCH_SIZE,
            "--checkpoint", f"{SMALL_NN_DIR}/{name}.pt",
            "--artifacts-dir", SMALL_NN_DIR,
            "--device", "cuda",
            "--resume",
        )
    finally:
        push_results()

### musdb7, remix

In [ ]:
small_nn("musdb7", remix=True)

### musdb7, no remix

In [ ]:
small_nn("musdb7", remix=False)

### musdb7_phone, remix

In [ ]:
small_nn("musdb7_phone", remix=True)

### musdb7_phone, no remix

In [ ]:
small_nn("musdb7_phone", remix=False)

### Optional: narrowband

Only if time is left: the hardest degradation (T3-06).

In [ ]:
# small_nn("musdb7_narrowband", remix=True)

### All loss curves

One figure with the training and validation losses of every finished run.

In [ ]:
logs = sorted(glob.glob(f"{SMALL_NN_DIR}/SmallMaskNet_*_log.csv"))
print("\n".join(logs))
python_module(
    "task3_uvr.plot_small_nn", *logs, "--output", "results/task3_small_nn_losses.png"
)
push_results()

from IPython.display import Image, display

display(Image(f"{REPO_DIR}/results/task3_small_nn_losses.png"))

## 7. Bring the results home

On Drive, `ml_tek5_checkpoints/task3/`:
- `results/`: `task3_sdr_clips.csv`, `task3_uvr.csv` and the loss figures.
  Copy them into the local `results/` folder and commit them.
- `small_nn/`: checkpoints (`.pt`), loss logs (`_log.csv`), ONNX models,
  their `_io.npz` and `model_data/<hash>.json`. These are not committed; the
  ONNX files and JSONs go to UVR for T3-09.

In [ ]:
!ls -la {DRIVE_RESULTS} {SMALL_NN_DIR}

## 8. T3-10: fine-tuning HQ 3 on degraded audio

Rerun sections 1-5 first (section 6 is not needed). One run fine-tunes
`UVR-MDX-NET-Inst_HQ_3` on `musdb7_phone` with lr 1e-6 and a 100-step
warmup. The loss is L1 on the accompaniment waveform (HQ 3's primary stem
is Instrumental), BatchNorm stays in eval mode, and every 100 steps the
validation loss is logged on `musdb7_phone` and on clean `musdb7`. At the
end the script exports the model (opset 17), checks it in onnxruntime 1.13.1
and measures the validation SDR before and after, on both datasets.

Checkpoints, logs, ONNX files and their `model_data/<hash>.json` go to
`ml_tek5_checkpoints/task3/finetune/` on Drive. A cut session resumes from
the last checkpoint (every 250 steps) when the cell is rerun.

**Upload once:** `UVR-MDX-NET-Inst_HQ_3.onnx` (from
`<UVR>/models/MDX_Net_Models/`) to `ml_tek5_checkpoints/task3/models/`.

In [ ]:
import torch

FINETUNE_DIR = f"{TASK3_DIR}/finetune"
HQ3_ONNX = f"{TASK3_DIR}/models/UVR-MDX-NET-Inst_HQ_3.onnx"
os.makedirs(FINETUNE_DIR, exist_ok=True)
if not os.path.isfile(HQ3_ONNX):
    raise FileNotFoundError(f"Upload UVR-MDX-NET-Inst_HQ_3.onnx to {TASK3_DIR}/models/ first")

gpu = torch.cuda.get_device_properties(0)
GPU_GB = gpu.total_memory / 1024**3
print(f"{gpu.name}, {GPU_GB:.1f} GB")

### onnxruntime 1.13.1 (UVR's version)

Every export is loaded in UVR's onnxruntime, 1.13.1, which has wheels only
up to Python 3.10. `uv` installs a separate Python 3.10 environment with it.
The script finds it through `ML_TEK5_ORT113_PYTHON`. Without it, the check
is skipped and the command to run it is printed.

In [ ]:
ORT113_DIR = "/content/ort113"
ORT113_PYTHON = f"{ORT113_DIR}/bin/python"
if not os.path.isfile(ORT113_PYTHON):
    !pip install -q uv
    !uv venv -q --python 3.10 {ORT113_DIR}
    !uv pip install -q --python {ORT113_PYTHON} onnxruntime==1.13.1 "numpy<2"
os.environ["ML_TEK5_ORT113_PYTHON"] = ORT113_PYTHON
!{ORT113_PYTHON} -c "import onnxruntime; print('onnxruntime', onnxruntime.__version__)"

### Run

On CPU one step of batch 1 needs about 5.4 GB. Batch 2 fits on a 16 GB GPU
(T4, P100) and batch 4 on a 40 GB A100. If CUDA runs out of memory, halve
`FT_BATCH_SIZE`: the run then gets another name, so it starts fresh rather
than resuming a checkpoint with other settings. 2,000 steps of batch 2 cover
the 80 training clips (9 minutes of audio) about 50 times.

The first run also measures the original HQ 3 on the 14 validation clips of
`musdb7_phone`. Its medians must match T3-07 (vocals 10.47 dB,
accompaniment 8.82 dB); otherwise the script prints a warning. `--report-test`
adds the test SDR: use it only for the final model, never to choose
settings.

In [ ]:
FT_STEPS = 2000
FT_LR = 1e-6
FT_BATCH_SIZE = 4 if GPU_GB > 30 else 2


def finetune(dataset="musdb7_phone", lr=FT_LR, steps=FT_STEPS,
             batch_size=FT_BATCH_SIZE, freeze="none", report_test=False):
    pull_results()
    try:
        python_module(
            "task3_uvr.finetune",
            "--dataset", dataset,
            "--model", HQ3_ONNX,
            "--lr", lr,
            "--steps", steps,
            "--batch-size", batch_size,
            "--freeze", freeze,
            "--artifacts-dir", FINETUNE_DIR,
            "--device", "cuda",
            "--resume",
            *(["--report-test"] if report_test else []),
        )
    finally:
        push_results()

### musdb7_phone, lr 1e-6

In [ ]:
finetune("musdb7_phone")

### Optional, once the first run is read

Only on the validation results: a larger learning rate if the validation
loss barely moved, or the decoder only if clean musdb7 got worse.

In [ ]:
# finetune("musdb7_phone", lr=1e-5)
# finetune("musdb7_phone", freeze="encoder")
# finetune("musdb7_narrowband")

### Bring the results home

`results/` on Drive now also holds `task3_finetune.json`,
`task3_finetune_before_after.csv` and `task3_finetune_loss_<run>.png`. Copy
them with the CSVs into the local `results/` folder and commit them.
`finetune/` holds the checkpoints, logs, ONNX files and their
`model_data/<hash>.json`. These are not committed; to listen in UVR, copy
the ONNX file to `<UVR>/models/MDX_Net_Models/` and the JSON to
`<UVR>/models/MDX_Net_Models/model_data/`.

In [ ]:
!ls -la {DRIVE_RESULTS} {FINETUNE_DIR}
import pandas as pd

pd.read_csv(f"{REPO_DIR}/results/task3_finetune_before_after.csv")